# Notebook 08 — Optimización de la posición del detector

Pregunta práctica: **¿dónde conviene poner el detector?** Para responderla, recorremos
un grid de posiciones candidatas alrededor del cráter, computamos el muograma DEM en cada
una, y elegimos la posición que maximiza una métrica de "información" — definida como la
varianza de la transmisión $T(\theta, \phi)$ sobre los píxeles del volcán.

La idea: si todas las direcciones son cielo abierto (T≈1) o todas opacas (T≈0), el
muograma no carga información tomográfica. La varianza alta indica contraste — más
señal útil.

**Costo**: ~1 s por posición × 11×11 = 121 posiciones ≈ 2 min wall-clock. Usamos malla
angular más gruesa (1° en vez de 0.5°) y `n_steps=200` para acelerar; la métrica es
robusta frente a esta sub-resolución.

In [8]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from pyproj import Transformer
from tqdm import tqdm

from analysis.muograma import (
    DEMTerrain, rock_opacity_grid, transmission_map, muograma_information,
)

In [9]:
# Target volcano + default station (same convention as NB 04 / 05)
TARGET_VOLCAN = "fuego"

with open(REPO_ROOT / "data" / "volcanoes.yaml") as f:
    VOLCAN_CFG = yaml.safe_load(f)[TARGET_VOLCAN]

DEM_FILE = REPO_ROOT / 'data' / 'dem' / 'Copernicus_DSM_COG_10_N14_00_W091_00_DEM.tif'

terrain = DEMTerrain(DEM_FILE,
                     origin_lonlat=(VOLCAN_CFG['lon'], VOLCAN_CFG['lat']),
                     density_gcc=VOLCAN_CFG['cone']['density_gcc'])
summit_z = terrain.summit_elevation

print(f"TARGET_VOLCAN = {TARGET_VOLCAN}")
print(f"  Cráter @ ({VOLCAN_CFG['lat']}, {VOLCAN_CFG['lon']}), cima a {summit_z:.0f} m")
print(f"  Default station (referencia): {VOLCAN_CFG['default_station']}")

TARGET_VOLCAN = fuego
  Cráter @ (14.473, -90.88), cima a 3672 m
  Default station (referencia): FG16


## 1. Grid de posiciones candidatas

Cuadrícula de 11×11 = 121 puntos en una ventana de ±10 km alrededor del cráter, en
coordenadas locales UTM (eje X = este, Y = norte, ambos en metros con cero en el cráter).
Cada celda mide 2 km × 2 km.

In [10]:
# Grid de posiciones candidatas (en coords locales relativas al cráter)
HALF_EXTENT_KM = 10.0
GRID_N = 11

xs_cand = np.linspace(-HALF_EXTENT_KM*1000, HALF_EXTENT_KM*1000, GRID_N)
ys_cand = np.linspace(-HALF_EXTENT_KM*1000, HALF_EXTENT_KM*1000, GRID_N)
XX, YY = np.meshgrid(xs_cand, ys_cand)

print(f"Grid: {GRID_N}×{GRID_N} = {GRID_N*GRID_N} posiciones candidatas")
print(f"Cobertura: ±{HALF_EXTENT_KM} km desde el cráter (paso {2*HALF_EXTENT_KM/(GRID_N-1):.1f} km)")

Grid: 11×11 = 121 posiciones candidatas
Cobertura: ±10.0 km desde el cráter (paso 2.0 km)


## 2. Setup de ray tracing (más gruesa para optimización rápida)

`theta_deg ∈ [40°, 85°]` paso 1° → 46 píxeles cenitales.
`phi_deg ∈ [0°, 360°)` paso 5° → 72 píxeles azimutales (rango completo para no asumir
dirección — la malla es del *espacio*, no del campo de visión del detector).
46 × 72 = 3312 píxeles por posición. ~10× más rápido que las mallas finas de NB 05.

In [11]:
theta_deg_opt = np.arange(40.0, 86.0, 1.0)
phi_deg_opt   = np.arange(0.0, 360.0, 5.0)
theta_rad_opt = np.radians(theta_deg_opt)
phi_rad_opt   = np.radians(phi_deg_opt)
print(f"Malla angular: {len(theta_deg_opt)}×{len(phi_deg_opt)} = {len(theta_deg_opt)*len(phi_deg_opt)} pixeles")

Malla angular: 46×72 = 3312 pixeles


## 3. Loop sobre posiciones — muograma + métrica

Por cada `(det_x, det_y)`: tomar la elevación del terreno + 2 m, calcular L_DEM,
calcular T(θ,φ), evaluar `muograma_information(T, L)` (varianza de T sobre píxeles
del volcán). Se guarda todo en `info_grid` (matriz 11×11).

In [12]:
info_grid = np.zeros((GRID_N, GRID_N))
det_z_grid = np.zeros((GRID_N, GRID_N))
L_max_grid = np.zeros((GRID_N, GRID_N))

print("Recorriendo grid de posiciones...")
for i in tqdm(range(GRID_N)):
    for j in range(GRID_N):
        det_x_ij = float(XX[i, j])
        det_y_ij = float(YY[i, j])
        # Elevación del terreno en esa posición + 2 m altura del detector
        det_z_ground = float(terrain.elevation_at(np.array(det_x_ij), np.array(det_y_ij)))
        det_z_ij = det_z_ground + 2.0
        det_z_grid[i, j] = det_z_ground

        detector_xyz = (det_x_ij, det_y_ij, det_z_ij)
        L = rock_opacity_grid(terrain, detector_xyz, theta_rad_opt, phi_rad_opt,
                              max_length_m=8_000.0, n_steps=200)
        T = transmission_map(L, theta_rad_opt)
        info_grid[i, j] = muograma_information(T, L, mode='variance_volcano')
        L_max_grid[i, j] = L.max()

print(f"\nValor de información — rango: [{info_grid.min():.4f}, {info_grid.max():.4f}]")
i_best, j_best = np.unravel_index(np.argmax(info_grid), info_grid.shape)
best_x, best_y = XX[i_best, j_best], YY[i_best, j_best]
print(f"\nMejor posición: ({best_x/1000:+.2f}, {best_y/1000:+.2f}) km del cráter")
print(f"  info = {info_grid[i_best, j_best]:.4f}")
print(f"  L_max = {L_max_grid[i_best, j_best]:.0f} g/cm²")
print(f"  altitud del terreno = {det_z_grid[i_best, j_best]:.0f} m")

Recorriendo grid de posiciones...


100%|██████████████████████████████████████████████████████████████████████████████████| 11/11 [00:30<00:00,  2.79s/it]


Valor de información — rango: [0.0080, 0.1955]

Mejor posición: (+8.00, -10.00) km del cráter
  info = 0.1955
  L_max = 996400 g/cm²
  altitud del terreno = 697 m


## 4. INSIVUMEH-RSN station ranking

All five RSN stations targeting Fuego are evaluated with the same angular grid and
information metric used for the spatial sweep. This identifies the best *real* candidate
site before comparing against the abstract grid optimum.

In [ ]:
# Evaluate all INSIVUMEH-RSN stations targeting Fuego
stations_fuego = sites_df[
    sites_df['target_volcan'] == TARGET_VOLCAN
].copy().reset_index(drop=True)

_tx_st = Transformer.from_crs("EPSG:4326", f"EPSG:{terrain.utm_epsg}", always_xy=True)

station_results = []
print(f"Evaluating {len(stations_fuego)} RSN stations for {TARGET_VOLCAN.title()}...\n")

for _, row in stations_fuego.iterrows():
    e_st, n_st = _tx_st.transform(row['lon'], row['lat'])
    sx = float(e_st - terrain.origin_utm[0])
    sy = float(n_st - terrain.origin_utm[1])
    sz_ground = float(terrain.elevation_at(np.array(sx), np.array(sy)))
    sz = sz_ground + 2.0

    L_st = rock_opacity_grid(terrain, (sx, sy, sz), theta_rad_opt, phi_rad_opt,
                             max_length_m=8_000.0, n_steps=200)
    T_st = transmission_map(L_st, theta_rad_opt)
    info_st = muograma_information(T_st, L_st, mode='variance_volcano')

    station_results.append({
        'name':     row['name'],
        'east_km':  sx / 1000,
        'north_km': sy / 1000,
        'elev_m':   sz_ground,
        'dist_km':  row['dist_km'],
        'info':     info_st,
        'L_max':    L_st.max(),
    })

station_results.sort(key=lambda x: x['info'], reverse=True)

# info_default used by heatmap cell below
info_default = next(r['info'] for r in station_results
                    if r['name'] == VOLCAN_CFG['default_station'])
best_rsn = station_results[0]

print(f"{'Rank':>4} {'Station':>8} {'East(km)':>9} {'N(km)':>7} "
      f"{'Elev(m)':>8} {'Info':>7} {'L_max(g/cm²)':>14}  Note")
print("-" * 74)
for k, r in enumerate(station_results, 1):
    note = " ← best RSN" if k == 1 else (
           " ← default"  if r['name'] == VOLCAN_CFG['default_station'] else "")
    print(f"{k:>4}  {r['name']:>7}  {r['east_km']:>+8.2f}  {r['north_km']:>+6.2f}  "
          f"{r['elev_m']:>7.0f}  {r['info']:>7.4f}  {r['L_max']:>13.0f}{note}")

print(f"\nGrid optimum : info={info_grid[i_best, j_best]:.4f}  "
      f"pos=({best_x/1000:+.1f}, {best_y/1000:+.1f}) km  (may not be accessible)")
print(f"Best RSN     : {best_rsn['name']}  info={best_rsn['info']:.4f}  "
      f"({best_rsn['info']/info_default:.2f}× over {VOLCAN_CFG['default_station']})")


## 5. Information score heatmap with RSN stations

DEM como fondo (hillshade), heatmap de información encima como overlay semitransparente.
El cráter va con estrella roja, el máximo de información con marker amarillo, la estación
default de INSIVUMEH con cuadrado cian (referencia para comparar).

In [ ]:
# DEM background ±15 km
ext = 15_000
xs_bg = np.linspace(-ext, ext, 500)
ys_bg = np.linspace(-ext, ext, 500)
XB, YB = np.meshgrid(xs_bg, ys_bg)
Z_bg = terrain.elevation_at(XB, YB)
ls_bg = LightSource(azdeg=315, altdeg=45)
rgb_bg = ls_bg.shade(Z_bg, cmap=plt.cm.terrain, vert_exag=2.0, blend_mode='overlay')

fig, ax = plt.subplots(figsize=(11, 10))
ax.imshow(rgb_bg, extent=[-ext/1000, ext/1000]*2, origin='lower')

# Info-score heatmap
heatmap = ax.pcolormesh(XX/1000, YY/1000, info_grid, cmap='magma',
                        alpha=0.65, shading='auto')
plt.colorbar(heatmap, ax=ax,
             label='Information score = std($T$) over volcano pixels', shrink=0.7)

# Crater
ax.scatter(0, 0, marker='*', s=420, color='red', edgecolor='black',
           lw=1.5, zorder=11, label=f'Crater of {TARGET_VOLCAN.title()}')

# Grid optimum
ax.scatter(best_x/1000, best_y/1000, marker='X', s=300, color='gold',
           edgecolor='black', lw=1.5, zorder=11,
           label=f'Grid optimum  I={info_grid[i_best,j_best]:.3f}')

# All RSN stations — marker size proportional to info score
info_vals = np.array([r['info'] for r in station_results])
sizes = 80 + 400 * (info_vals - info_vals.min()) / (info_vals.max() - info_vals.min() + 1e-9)
colors = ['cyan', 'deepskyblue', 'dodgerblue', 'steelblue', 'royalblue']

for k, (r, sz_mk, col) in enumerate(zip(station_results, sizes, colors)):
    ax.scatter(r['east_km'], r['north_km'], marker='s', s=sz_mk,
               color=col, edgecolor='black', lw=1.2, zorder=10)
    ax.annotate(f"{r['name']}\nI={r['info']:.3f}",
                (r['east_km'], r['north_km']),
                xytext=(6, 6), textcoords='offset points',
                fontsize=8, color='white',
                bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.5))

# Dummy handle for legend
from matplotlib.lines import Line2D
rsn_handle = Line2D([0], [0], marker='s', color='w', markerfacecolor='cyan',
                    markeredgecolor='black', markersize=9,
                    label='RSN stations (size ∝ info score)')
ax.legend(handles=ax.get_legend_handles_labels()[0] + [rsn_handle],
          loc='upper left', fontsize=9)

ax.set_xlabel('Easting from crater [km]')
ax.set_ylabel('Northing from crater [km]')
ax.set_title(
    f'Information score vs detector position — {TARGET_VOLCAN.title()}\n'
    f'RSN station ranking: '
    + '  >  '.join(f"{r['name']}({r['info']:.3f})" for r in station_results)
)
ax.set_aspect('equal')
ax.set_xlim(-ext/1000, ext/1000)
ax.set_ylim(-ext/1000, ext/1000)
plt.tight_layout()
fig.savefig(REPO_ROOT / 'docs/paper_volcanica/figures' / f'fig_detector_optim_{TARGET_VOLCAN}.png',
            bbox_inches='tight', dpi=300)
plt.show()


## 6. Summary: RSN stations vs grid optimum

¿Cuánta mejora ofrece el óptimo respecto a la estación default? Y ¿es esa posición
geográficamente accesible? Ojo: el óptimo lo elige *únicamente* la métrica matemática,
sin saber nada de logística (carreteras, propiedad de la tierra, peligro volcánico).

In [ ]:
# Summary: all RSN stations vs grid optimum
info_best = float(info_grid[i_best, j_best])

print("=" * 60)
print("INSIVUMEH-RSN station ranking (best to worst)")
print("=" * 60)
for k, r in enumerate(station_results, 1):
    ratio = r['info'] / info_default
    print(f"  {k}. {r['name']:5s}  info={r['info']:.4f}  "
          f"({ratio:.2f}× over {VOLCAN_CFG['default_station']})  "
          f"L_max={r['L_max']:.0f} g/cm²")

print()
print(f"Grid optimum (abstract): info={info_best:.4f}  "
      f"pos=({best_x/1000:+.1f}, {best_y/1000:+.1f}) km  "
      f"z={det_z_grid[i_best,j_best]:.0f} m")
print(f"  → {info_best/info_default:.2f}× over {VOLCAN_CFG['default_station']}")
print()
print("Notes:")
print("- Grid optimum is at the SE boundary → true math. optimum may lie further SE.")
print("- Best RSN station within the accessible area is the recommended deployment site.")
print("- Stations co-located (FG12/FG3) will give near-identical scores.")


## Notas para extender

- **Grid más fino**: cambiá `GRID_N` o `HALF_EXTENT_KM`. El cómputo escala como N².
- **Otra métrica**: `muograma_information(..., mode='fraction_blocked')` u `'mean_deviation'`.
- **Múltiples detectores**: la información combinada de 2 detectores no es la suma — habría que computar una métrica conjunta tipo entropía mutua, no implementado acá.
- **Restricciones de accesibilidad**: enmascarar el `info_grid` con un mapa de áreas accesibles (carreteras, tierra de INSIVUMEH/INAB, etc.) antes de tomar el argmax.